# 05 — Visualizaciones (Figuras 1-7)
**IA Generativa y Saber Pro 2021-2024** | Eduardo A. Victoria Cadena

Genera las 7 figuras del análisis: boxplots, histogramas, dispersión distancia, coefplots β_IA, brechas departamentales, mapa de calor de correlaciones y serie temporal.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUTA_PROYECTO = '/content/drive/MyDrive/IA-Y-EDUCACION-SUPERIOR'
import sys; sys.path.insert(0, RUTA_PROYECTO + '/python')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')
from utils import (MODULOS_GENERICOS, ETIQUETAS_VARS, COLORES_DEPTO,
                   ALPHA, guardar_figura)

plt.rcParams.update({'figure.dpi':150, 'font.size':10,
                     'axes.spines.top':False, 'axes.spines.right':False})

PROC_DIR = f'{RUTA_PROYECTO}/data/processed'
TBL_DIR  = f'{RUTA_PROYECTO}/outputs/tablas'
FIG_DIR  = f'{RUTA_PROYECTO}/outputs/figuras'
os.makedirs(FIG_DIR, exist_ok=True)

df      = pd.read_pickle(f'{PROC_DIR}/datos_analisis.pkl')
tabla4  = pd.read_csv(f'{TBL_DIR}/T4_coeficientes_todos_modelos.csv')
print("Datos cargados:", len(df), "filas")

In [ ]:
# ── Fig 1: Boxplot 12 cajas (2 períodos × 6 departamentos) ──────────────
fig, ax = plt.subplots(figsize=(13, 6))

orden_depto = list(COLORES_DEPTO.keys())
df_p = df[df['depto_ies'].isin(orden_depto) & df['puntaje_saberpro_generico'].notna()].copy()
df_p['periodo_lbl'] = df_p['periodo_ia'].map({0:'2021-2022',1:'2023-2024'})
df_p['depto_lbl']   = pd.Categorical(df_p['depto_ies'], categories=orden_depto, ordered=True)

sns.boxplot(data=df_p, x='depto_lbl', y='puntaje_saberpro_generico',
            hue='periodo_lbl',
            palette={'2021-2022':'#90CAF9','2023-2024':'#EF9A9A'},
            flierprops=dict(marker='o', markersize=2, alpha=0.3),
            width=0.65, ax=ax)

# Etiquetas n
for i, depto in enumerate(orden_depto):
    for j, per in enumerate([0, 1]):
        n = len(df_p[(df_p['depto_ies']==depto) & (df_p['periodo_ia']==per)])
        ax.text(i + (j-0.5)*0.42, ax.get_ylim()[0] + 2,
                f'n={n}', ha='center', va='bottom', fontsize=7, color='gray')

ax.set(title='Figura 1. Puntaje Saber Pro genérico por departamento y período',
       xlabel='Departamento de la IES',
       ylabel='Puntaje genérico (promedio 5 módulos)')
ax.legend(title='Período', loc='upper right')
plt.tight_layout()
guardar_figura(fig, 'F1_boxplot_periodo_depto', FIG_DIR)
plt.show()

In [ ]:
# ── Fig 2: Histograma comparativo por período ────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
colores = {'2021-2022':'#1565C0','2023-2024':'#B71C1C'}
for per, lbl in [(0,'2021-2022'),(1,'2023-2024')]:
    datos = df[df['periodo_ia']==per]['puntaje_saberpro_generico'].dropna()
    media = datos.mean()
    ax.hist(datos, bins=40, density=True, alpha=0.45, color=colores[lbl], label=lbl)
    from scipy.stats import gaussian_kde
    kde = gaussian_kde(datos)
    x   = np.linspace(datos.min(), datos.max(), 300)
    ax.plot(x, kde(x), color=colores[lbl], linewidth=1.8)
    ax.axvline(media, color=colores[lbl], linestyle='--', linewidth=1.3)
    ax.text(media+0.5, ax.get_ylim()[1]*0.9 - list(colores.keys()).index(lbl)*0.06,
            f'$\bar{{x}}$={media:.1f}', color=colores[lbl], fontsize=9)
ax.set(title='Figura 2. Distribución del puntaje genérico Saber Pro por período',
       xlabel='Puntaje genérico', ylabel='Densidad')
ax.legend(title='Período')
plt.tight_layout()
guardar_figura(fig, 'F2_histograma_periodos', FIG_DIR)
plt.show()

In [ ]:
# ── Fig 3: Dispersión distancia vs puntaje promedio por IES ──────────────
df_ies = (df[df['distancia_bogota_km'].notna() & df['puntaje_saberpro_generico'].notna()]
          .groupby(['nombre_ies','depto_ies','distancia_bogota_km','periodo_ia'])
          .agg(puntaje_medio=('puntaje_saberpro_generico','mean'),
               n=('puntaje_saberpro_generico','count'))
          .reset_index())
df_ies['periodo_lbl'] = df_ies['periodo_ia'].map({0:'2021-2022',1:'2023-2024'})

fig, ax = plt.subplots(figsize=(11, 6))
markers = {'2021-2022':'o','2023-2024':'^'}
for per_lbl, grp in df_ies.groupby('periodo_lbl'):
    for _, row in grp.iterrows():
        color = COLORES_DEPTO.get(row['depto_ies'],'gray')
        ax.scatter(row['distancia_bogota_km'], row['puntaje_medio'],
                   color=color, marker=markers[per_lbl],
                   s=row['n']*0.15 + 40, alpha=0.85)

# Línea de tendencia OLS
from numpy.polynomial.polynomial import polyfit
x_all = df_ies['distancia_bogota_km'].values
y_all = df_ies['puntaje_medio'].values
coefs = np.polyfit(x_all, y_all, 1)
x_lin = np.linspace(0, 800, 200)
ax.plot(x_lin, np.polyval(coefs, x_lin), 'k--', linewidth=1.2, label='Tendencia OLS')

# Leyenda departamentos
from matplotlib.lines import Line2D
leyenda_depto  = [Line2D([0],[0],marker='o',color='w',markerfacecolor=c,
                          markersize=9,label=d) for d,c in COLORES_DEPTO.items()]
leyenda_periodo = [Line2D([0],[0],marker=m,color='gray',
                           label=l,linestyle='none',markersize=8)
                   for l,m in markers.items()]
ax.legend(handles=leyenda_depto+leyenda_periodo, loc='upper right',
          fontsize=8, ncol=2, title='Dpto. / Período')

ax.set_xticks([0,210,310,415,460,785])
ax.set_xticklabels(['Bogotá\n(0km)','Tolima\n(210)','Huila\n(310)',
                    'Antioquia\n(415)','Valle\n(460)','Nariño\n(785)'], fontsize=9)
ax.set(title='Figura 3. Puntaje Saber Pro vs Distancia a Bogotá por IES',
       xlabel='Distancia a Bogotá D.C. (km vía terrestre)',
       ylabel='Puntaje genérico promedio por IES')
plt.tight_layout()
guardar_figura(fig, 'F3_dispersion_distancia', FIG_DIR)
plt.show()

In [ ]:
# ── Fig 4: Coefplot β_IA por módulo y especificación ─────────────────────
df_beta = (tabla4[tabla4['variable']=='periodo_ia']
           .assign(modulo=lambda x: x['var_dep'].map(ETIQUETAS_VARS).fillna(x['var_dep']),
                   ic_l=lambda x: x['coef']-1.96*x['se'],
                   ic_u=lambda x: x['coef']+1.96*x['se'],
                   sig=lambda x: x['p_valor'] < ALPHA)
           .sort_values('var_dep'))

specs   = df_beta['spec'].unique()
palette = {'base':'#1976D2','ef_ies':'#388E3C','ef_mun':'#F57C00'}
labels  = {'base':'Especificación base','ef_ies':'+ EF por IES','ef_mun':'+ EF por municipio'}
modulos = df_beta['modulo'].unique()
y_pos   = {m: i for i, m in enumerate(modulos)}
offsets = {s: j*0.25-0.25 for j, s in enumerate(specs)}

fig, ax = plt.subplots(figsize=(10, 6))
for sp in specs:
    sub = df_beta[df_beta['spec']==sp]
    for _, row in sub.iterrows():
        y = y_pos[row['modulo']] + offsets[sp]
        color = palette.get(sp,'gray')
        ax.errorbar(row['coef'], y,
                    xerr=[[row['coef']-row['ic_l']],[row['ic_u']-row['coef']]],
                    fmt='o' if row['sig'] else 's',
                    color=color, capsize=4, markersize=7, linewidth=1.5)

ax.axvline(0, color='gray', linestyle='--', linewidth=1)
ax.set_yticks(list(y_pos.values()))
ax.set_yticklabels(list(y_pos.keys()), fontsize=9)
from matplotlib.lines import Line2D
leyenda = [Line2D([0],[0],color=palette[s],marker='o',linestyle='-',
                   markersize=7,label=labels[s]) for s in specs]
ax.legend(handles=leyenda, title='Especificación', fontsize=9)
ax.set(title='Figura 4. Coeficiente β_IA por módulo y especificación',
       xlabel='β_IA (diferencia condicional de puntaje)',
       ylabel=None)
ax.text(0.02, -0.12, 'IC 95% con errores clusterizados por IES. Fuente: DataICFES.',
        transform=ax.transAxes, fontsize=8, color='gray')
plt.tight_layout()
guardar_figura(fig, 'F4_coefplot_beta_ia', FIG_DIR)
plt.show()

In [ ]:
# ── Fig 5: Brechas departamentales condicionales vs Bogotá ───────────────
depto_vars   = ['d_antioquia','d_valle','d_huila','d_narino','d_tolima']
depto_labels = {'d_antioquia':'Antioquia','d_valle':'Valle del Cauca',
                'd_huila':'Huila','d_narino':'Nariño','d_tolima':'Tolima'}

df_dep = (tabla4[(tabla4['variable'].isin(depto_vars)) & (tabla4['spec']=='base')]
          .assign(depto=lambda x: x['variable'].map(depto_labels),
                  modulo=lambda x: x['var_dep'].map(ETIQUETAS_VARS).fillna(x['var_dep']),
                  ic_l=lambda x: x['coef']-1.96*x['se'],
                  ic_u=lambda x: x['coef']+1.96*x['se'],
                  sig=lambda x: x['p_valor']<ALPHA))

fig, ax = plt.subplots(figsize=(11, 6))
modulos_uniq = df_dep['modulo'].unique()
pal = sns.color_palette('tab10', len(modulos_uniq))
col_map = dict(zip(modulos_uniq, pal))

deptos_uniq = ['Nariño','Huila','Tolima','Valle del Cauca','Antioquia']
y_pos = {d: i for i, d in enumerate(deptos_uniq)}
off_step = 0.12
for k, mod_lbl in enumerate(modulos_uniq):
    sub = df_dep[df_dep['modulo']==mod_lbl]
    for _, row in sub.iterrows():
        if row['depto'] not in y_pos: continue
        y = y_pos[row['depto']] + (k - len(modulos_uniq)/2) * off_step
        ax.errorbar(row['coef'], y,
                    xerr=[[row['coef']-row['ic_l']],[row['ic_u']-row['coef']]],
                    fmt='o' if row['sig'] else 's',
                    color=col_map[mod_lbl], capsize=3,
                    markersize=5, linewidth=1.2, alpha=0.85)

ax.axvline(0, color='black', linestyle='--', linewidth=1.2)
ax.set_yticks(list(y_pos.values()))
ax.set_yticklabels(list(y_pos.keys()), fontsize=10)
from matplotlib.lines import Line2D
leyenda = [Line2D([0],[0],color=col_map[m],marker='o',linestyle='-',
                   markersize=6,label=m) for m in modulos_uniq]
ax.legend(handles=leyenda, title='Módulo', fontsize=8, loc='lower right')
ax.set(title='Figura 5. Brechas departamentales condicionales vs Bogotá D.C.',
       xlabel='Diferencia condicional en puntaje respecto a Bogotá D.C.')
plt.tight_layout()
guardar_figura(fig, 'F5_brechas_departamentales', FIG_DIR)
plt.show()

In [ ]:
# ── Fig 6: Mapa de calor de correlaciones ────────────────────────────────
vars_corr = ['puntaje_saberpro_generico'] + [m for m in MODULOS_GENERICOS if m in df.columns]
vars_corr += ['puntaje_saber11','estrato','nivel_educ_padre',
              'distancia_bogota_km','genero','internet','estu_trabaja']
vars_corr = [v for v in vars_corr if v in df.columns]

mat_corr = df[vars_corr].rename(columns=ETIQUETAS_VARS).corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(mat_corr, dtype=bool))
sns.heatmap(mat_corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            annot_kws={'size':7}, linewidths=0.4, cbar_kws={'shrink':0.8})
ax.set(title='Figura 6. Mapa de calor — Correlaciones entre variables del modelo')
plt.xticks(fontsize=7); plt.yticks(fontsize=7)
plt.tight_layout()
guardar_figura(fig, 'F6_correlaciones', FIG_DIR)
plt.show()

In [ ]:
# ── Fig 7: Serie temporal por departamento (2021-2024) ───────────────────
df_anual = (df[df['puntaje_saberpro_generico'].notna()]
            .groupby(['ANIO_APLICACION','depto_ies'])
            .agg(media=('puntaje_saberpro_generico','mean'),
                 n=('puntaje_saberpro_generico','count'))
            .reset_index())

fig, ax = plt.subplots(figsize=(10, 6))
for depto, grp in df_anual.groupby('depto_ies'):
    color = COLORES_DEPTO.get(depto, 'gray')
    grp = grp.sort_values('ANIO_APLICACION')
    ax.plot(grp['ANIO_APLICACION'], grp['media'],
            marker='o', color=color, linewidth=2, label=depto,
            markersize=grp['n']/grp['n'].max()*12+3)

ax.axvline(2022.5, color='black', linestyle=':', linewidth=1.4)
ax.text(2022.55, ax.get_ylim()[1], 'Adopción masiva IA →',
        fontsize=8.5, va='top', color='black')
ax.set_xticks([2021,2022,2023,2024])
ax.set(title='Figura 7. Evolución del puntaje Saber Pro genérico por departamento (2021-2024)',
       xlabel='Año de aplicación',
       ylabel='Puntaje genérico promedio')
ax.legend(title='Departamento', fontsize=9)
plt.tight_layout()
guardar_figura(fig, 'F7_serie_anual_depto', FIG_DIR)
plt.show()
print("\n✅ Todas las figuras generadas y guardadas en outputs/figuras/")